# Vacancy Energy Feature Importance

RF permutation importance for $E_b$, $E_s$, and $E_s - E_b$.

Features are selected when `importance_perm_mean > importance_perm_std`
(and optional absolute threshold). Near-duplicates (|Pearson r| > 0.99) are deduplicated across feature families.

Includes solution thermodynamic features (`mu_dopant_eV`, MP hull metadata) and Hume–Rothery-style elemental deltas (`delta_r_vs_Li_pm`, `delta_valency_vs_Li`, `delta_chi_vs_Li`) when present.

**Data pool:** `OPT_ONLY = False` uses `_pre` (full dopant set); `True` uses `_opt` (lattice-optimized set with 2NN neighbour positions for solute–vacancy binding). Outputs are tagged `_pre` / `_opt`.


In [ ]:
from pathlib import Path
import warnings
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

from build_datasheet import (
    dedupe_correlated_features,
    discover_ml_feature_columns,
    feature_dedup_family,
    load_ml_frame,
)

warnings.filterwarnings(
    'ignore',
    message='Skipping features without any observed values',
    category=UserWarning,
    module='sklearn.impute._base',
)


In [ ]:
from matplotlib import font_manager as fm
from pathlib import Path as _Path
for _fp in (
    _Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'),
    _Path('/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf'),
):
    if _fp.is_file():
        fm.fontManager.addfont(str(_fp))
        break


In [ ]:
np.set_printoptions(precision=5)


In [ ]:
ML_DIR = Path('.').resolve()
REPO_ROOT = ML_DIR
DATA_DIR = ML_DIR / 'data'
FIGURES_DIR = ML_DIR / 'figures'

# False → full _pre set; True → lattice-optimized _opt set (2NN binding positions).
OPT_ONLY = False
ENERGY_POOL = 'opt' if OPT_ONLY else 'pre'
POOL_TAG = ENERGY_POOL
POOL_SUFFIX = f'_{POOL_TAG}'

merged = load_ml_frame(REPO_ROOT, energy_pool=ENERGY_POOL)
hume = [c for c in ('delta_r_vs_Li_pm', 'delta_valency_vs_Li', 'delta_chi_vs_Li') if c in merged.columns]
print(
    f'pool={POOL_TAG} (OPT_ONLY={OPT_ONLY}): samples={len(merged)}, '
    f'dopants={sorted(merged["element"].astype(str).tolist())}'
)
print(f'Hume–Rothery delta_* columns: {hume if hume else "none (check data/elemental_props_vs_li.csv)"}')


In [ ]:
TARGETS = {
    'binding': 'vac_binding_energy_eV',
    'solution': 'solution_energy_eV',
    'solution_minus_binding': 'E_solution_minus_binding_eV',
}
TARGET_LABELS = {
    'binding': r'$E_{\mathrm{b}}$',
    'solution': r'$E_{\mathrm{s}}$',
    'solution_minus_binding': r'$E_{\mathrm{eff}}$',
}
merged['E_solution_minus_binding_eV'] = (
    merged['solution_energy_eV'].astype(float) - merged['vac_binding_energy_eV'].astype(float)
)

feature_cols = discover_ml_feature_columns(merged, ML_DIR)
X = merged[feature_cols]
print(f'samples={len(X)}, features={len(feature_cols)}')

def clean_feature_name(name: str) -> str:
    s = str(name)
    if s.startswith('endmember_MagpieData'):
        rest = s[len('endmember_MagpieData'):].lstrip(' _')
        return f'endmember {rest}' if rest else 'endmember'
    if s.startswith('MagpieData'):
        return s[len('MagpieData'):].lstrip(' _')
    # Distinguish endmember metadata labels from unary Magpie names.
    if s.startswith('endmember_'):
        rest = s[len('endmember_'):].lstrip(' _')
        return f'endmember {rest}' if rest else 'endmember'
    return s

def format_plot_number(val, fmt='.2f') -> str:
    """Mathtext minus for vertically centered negative Pearson r in heatmaps."""
    if not np.isfinite(val):
        return ''
    return f'${val:{fmt}}$'

def feature_meaning(name: str) -> str:
    if str(name).startswith('endmember_MagpieData'):
        return f'Endmember Magpie descriptor: {clean_feature_name(name)}.'
    if str(name).startswith('MagpieData'):
        return f'Magpie descriptor: {clean_feature_name(name)}.'
    return f'Feature: {name}.'


In [ ]:
CORR_DEDUP_THRESHOLD = 0.99
REQUIRE_PERM_GT_STD = True
IMPORTANCE_THRESHOLD = 1e-3
MIN_FEATURES = 5
MAX_FEATURES = 50

def _pearson_r(X, a, b):
    pair = X[[a, b]].apply(pd.to_numeric, errors='coerce').dropna()
    return float(pair.corr().iloc[0, 1]) if len(pair) >= 2 else np.nan

def mark_significant(df):
    out = df.copy()
    out['significant'] = out['importance_perm_mean'] > out['importance_perm_std']
    return out

def select_features_for_analysis(importance_df):
    df = mark_significant(importance_df)
    df['feature_clean'] = df['feature'].map(clean_feature_name)
    mask = df['importance_perm_mean'] >= IMPORTANCE_THRESHOLD
    if REQUIRE_PERM_GT_STD:
        mask &= df['significant']
    sel = df[mask]
    if len(sel) < MIN_FEATURES:
        sel = df.head(MIN_FEATURES)
    elif len(sel) > MAX_FEATURES:
        sel = df.head(MAX_FEATURES)
    return sel.copy()

def usable_feature_cols(X: pd.DataFrame, cols: list[str]) -> list[str]:
    usable = []
    for col in cols:
        if col not in X.columns:
            continue
        if pd.to_numeric(X[col], errors='coerce').notna().any():
            usable.append(col)
    return usable


def compute_feature_importance(X, y, cols, n_estimators=600, n_repeats=40, random_state=42):
    fit_cols = usable_feature_cols(X, cols)
    if not fit_cols:
        raise ValueError('No features with observed values for importance analysis.')
    skipped = [c for c in cols if c not in fit_cols]
    if skipped:
        print(f'  skipping all-NaN features: {skipped}')
    X_fit = X[fit_cols].apply(pd.to_numeric, errors='coerce')
    model = Pipeline([
        ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
        ('rf', RandomForestRegressor(n_estimators=n_estimators, random_state=random_state, n_jobs=-1, max_features='sqrt')),
    ])
    cv = KFold(n_splits=5, shuffle=True, random_state=random_state)
    cv_r2 = cross_val_score(model, X_fit, y, cv=cv, scoring='r2')
    model.fit(X_fit, y)
    perm = permutation_importance(model, X_fit, y, n_repeats=n_repeats, random_state=random_state, scoring='r2', n_jobs=-1)
    df = pd.DataFrame({
        'feature': fit_cols,
        'importance_mdi': model.named_steps['rf'].feature_importances_,
        'importance_perm_mean': perm.importances_mean,
        'importance_perm_std': perm.importances_std,
    })
    df = mark_significant(df).sort_values('importance_perm_mean', ascending=False).reset_index(drop=True)
    return df, model, float(cv_r2.mean()), float(r2_score(y, model.predict(X_fit)))

importance_by_target = {}
for target_key, target_col in TARGETS.items():
    y = merged[target_col].astype(float)
    full_df, _, _, _ = compute_feature_importance(X, y, feature_cols)
    kept, dropped = dedupe_correlated_features(
        X, full_df, threshold=CORR_DEDUP_THRESHOLD, across_families=True
    )
    n_cross = sum(
        1
        for feat, keeper, _ in dropped
        if feature_dedup_family(feat) != feature_dedup_family(keeper)
    )
    n_endmember_in = sum(str(c).startswith('endmember_') for c in feature_cols)
    n_endmember_kept = sum(str(c).startswith('endmember_') for c in kept)
    imp_df, _, cv_r2, train_r2 = compute_feature_importance(X[kept], y, kept)
    n_endmember_sig = int(
        imp_df.loc[
            imp_df['feature'].astype(str).str.startswith('endmember_') & imp_df['significant']
        ].shape[0]
    )
    importance_by_target[target_key] = imp_df
    n_sig = int(imp_df['significant'].sum())
    path = DATA_DIR / f'{target_key}_feature_importance{POOL_SUFFIX}.csv'
    imp_df.to_csv(path, index=False)
    # Also write untagged copy when opt-only (legacy filename for model_train).
    if OPT_ONLY:
        legacy = DATA_DIR / f'{target_key}_feature_importance.csv'
        imp_df.to_csv(legacy, index=False)
    print(
        f"[{POOL_TAG}] {TARGET_LABELS[target_key]}: dedup {len(kept)}/{len(feature_cols)} "
        f"(dropped {len(dropped)}, cross-family {n_cross}; "
        f"endmember kept {n_endmember_kept}/{n_endmember_in}, significant endmember={n_endmember_sig}), "
        f"significant={n_sig}, CV R2={cv_r2:.3f} -> {path}"
    )


In [ ]:
colors_rgb = {'R': (220, 20, 60), 'G': (78, 162, 78), 'B': (0, 0, 135)}
alpha = 0.3
colors_hist = {k: tuple(list(float(x)/255 for x in v) + [alpha]) for k, v in colors_rgb.items()}
colors_blend_list = [np.array(c) for c in colors_hist.values()]
colors_list = [np.array(tuple(v/255 for v in rgb)) for rgb in colors_rgb.values()]
title_names = ['binding energy', 'solution energy', 'vacancy effective energy']
FONTSIZE = 12
TOP_N = 20
TICK_LENGTH = 6
TICK_WIDTH = 1.0
SAVE_FIGURES = True
PLOT_SHOW_TITLES = False
PLOT_SHOW_GRID = False
PLOT_SPINE_COLOR = 'black'
PLOT_SAVEFIG_FORMAT = 'pdf'
PLOT_SAVEFIG_DPI = 600
cmap_custom = mcolors.LinearSegmentedColormap.from_list('custom_cmap', ['darkblue', 'white', 'crimson'], N=256)
if SAVE_FIGURES:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
for target_key, importance_df, idx in zip(importance_by_target, importance_by_target.values(), range(3)):
    selected = select_features_for_analysis(importance_df)
    plot_df = selected.sort_values('importance_perm_mean')
    n = len(plot_df)
    print(f"[{POOL_TAG}] {target_key}: plotting {n} features (threshold={IMPORTANCE_THRESHOLD}, perm>std={REQUIRE_PERM_GT_STD})")

    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.28 * n)))
    ax.barh(plot_df['feature_clean'], plot_df['importance_perm_mean'], facecolor=colors_blend_list[idx], edgecolor=colors_list[idx], linewidth=1.5)
    ax.set_xlabel('Permutation importance (ΔR²)')
    if PLOT_SHOW_TITLES:
        ax.set_title(f'Significant features for {title_names[idx]}')
    ax.tick_params(axis='both', which='major', length=TICK_LENGTH, width=TICK_WIDTH)
    ax.grid(PLOT_SHOW_GRID, axis='x')
    ax.margins(y=0.01)
    for spine in ax.spines.values():
        spine.set_color(PLOT_SPINE_COLOR)
    fig.tight_layout()
    if SAVE_FIGURES:
        p = FIGURES_DIR / f'feature_importance_{target_key}{POOL_SUFFIX}.{PLOT_SAVEFIG_FORMAT}'
        fig.savefig(p, dpi=PLOT_SAVEFIG_DPI, bbox_inches='tight'); print('Saved', p)
    plt.show()

    feats = plot_df['feature'].tolist()
    if len(feats) < 2:
        continue
    labels = plot_df['feature_clean'].tolist()
    corr = X[feats].corr()
    corr.columns = labels; corr.index = labels
    fig, ax = plt.subplots(figsize=(max(8, 0.45*len(labels)), max(8, 0.45*len(labels))))
    annot_font = max(FONTSIZE - 5.1, 6) if len(labels) > 20 else FONTSIZE - 2
    annot = corr.map(lambda v: format_plot_number(v, '.2f') if np.isfinite(v) else '')

    im = sns.heatmap(
        corr,
        mask=np.triu(np.ones_like(corr, dtype=bool), k=1),
        cmap=cmap_custom,
        vmin=-1,
        vmax=1,
        square=True,
        annot=annot,
        fmt='',
        annot_kws={'size': annot_font},
        cbar=False,
        ax=ax,
    )
    
    cbar = fig.colorbar(
        im.get_children()[0],
        ax=ax,
        use_gridspec=True,
        shrink=0.76,
        aspect=34,
        pad=0.02
    )
    cbar.set_label(label='Pearson r', fontsize=FONTSIZE, labelpad=5)
    cbar.ax.tick_params(labelsize=FONTSIZE - 1)
    
    if PLOT_SHOW_TITLES:
        ax.set_title(f'Feature correlation for {title_names[idx]}')
    ax.tick_params(axis='both', which='major', length=TICK_LENGTH, width=TICK_WIDTH)
    ax.tick_params(axis='x', labelsize=FONTSIZE - 1) 
    ax.tick_params(axis='y', labelsize=FONTSIZE - 1) 
    for spine in ax.spines.values():
        spine.set_color(PLOT_SPINE_COLOR)
    fig.tight_layout()
    if SAVE_FIGURES:
        p = FIGURES_DIR / f'feature_correlation_matrix_{target_key}{POOL_SUFFIX}.{PLOT_SAVEFIG_FORMAT}'
        fig.savefig(p, dpi=PLOT_SAVEFIG_DPI, bbox_inches='tight'); print('Saved', p)
    plt.show()


In [ ]:
TOP_N_CORR = TOP_N

def top_significant_names(importance_df, top_n):
    sig = importance_df[importance_df['significant']] if REQUIRE_PERM_GT_STD else importance_df
    return sig.head(top_n)['feature'].astype(str).tolist()

unique_features = []
seen = set()
for tk in TARGETS:
    for f in top_significant_names(importance_by_target[tk], TOP_N_CORR):
        if f not in seen:
            seen.add(f); unique_features.append(f)
print(f'[{POOL_TAG}] Union of top-{TOP_N_CORR} significant features: {len(unique_features)}')

rows = []
for feat in unique_features:
    row = {'feature': feat, 'feature_clean': clean_feature_name(feat), 'feature_meaning': feature_meaning(feat)}
    for tk, col in TARGETS.items():
        x = merged[feat].astype(float); y = merged[col].astype(float)
        m = x.notna() & y.notna()
        r, p = pearsonr(x[m], y[m]) if m.sum() >= 3 else (np.nan, np.nan)
        row[f'pearson_r_{tk}'] = r; row[f'p_value_{tk}'] = p
        pm = importance_by_target[tk].loc[importance_by_target[tk]['feature']==feat, 'importance_perm_mean']
        row[f'perm_importance_{tk}'] = float(pm.iloc[0]) if len(pm) else np.nan
    rows.append(row)

pearson_corr_df = pd.DataFrame(rows)
pearson_corr_df.to_csv(DATA_DIR / f'top_feature_pearson_correlations{POOL_SUFFIX}.csv', index=False)
pivot = pearson_corr_df.set_index('feature_clean')[[c for c in pearson_corr_df.columns if c.startswith('pearson_r_')]]
pivot.columns = [c.replace('pearson_r_','') for c in pivot.columns]
pivot = pivot.rename(columns=lambda c: TARGET_LABELS.get(c,c))

fig, ax = plt.subplots(figsize=(8, max(4, 0.45*len(pivot))))
values = pivot.to_numpy(dtype=float)
im = ax.imshow(values, cmap=cmap_custom, vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=FONTSIZE + 2)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=FONTSIZE)
ax.tick_params(axis='both', which='major', length=TICK_LENGTH, width=TICK_WIDTH)
# ax.set_title('Pearson correlation vs targets', fontsize=FONTSIZE)

for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        val = values[i, j]
        if np.isfinite(val):
            ax.text(j, i, format_plot_number(val, '.2f'), ha='center', va='center', fontsize=FONTSIZE )


cbar = fig.colorbar(im, ax=ax, use_gridspec=True)
cbar.set_label(label='Pearson r', fontsize=FONTSIZE, labelpad=5) 
cbar.ax.tick_params(labelsize=FONTSIZE - 1 )

fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(FIGURES_DIR / f'pearson_correlations_top_features{POOL_SUFFIX}.{PLOT_SAVEFIG_FORMAT}', dpi=PLOT_SAVEFIG_DPI, bbox_inches='tight')
plt.show()
display(pearson_corr_df)

